# Scraping produc category
## Category level 1

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from lxml import etree
import time
from urllib.parse import urlparse, parse_qs

In [11]:
def get_category_lv1_title(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'th-TH,th;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.naiin.com/',
    }

    parsed_url = urlparse(url)
    params = parse_qs(parsed_url.query)
    p_id = params.get('product_type_id', [''])[0]
    cat1_code = params.get('category_1_code', [''])[0]

    # เตรียมข้อมูลพื้นฐานรอไว้ก่อน
    data = {
        'product_type_id': p_id,
        'category_1_code': cat1_code,
        'category_1_title': "-", # Default เป็น "-"
        'status_code': None,
        'url': url
    }

    try:
        response = session.get(url, headers=headers, timeout=15)
        data['status_code'] = response.status_code

        # ตรวจสอบว่าถ้าเป็น 200 OK ถึงจะทำการ Parse หา Title
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            h1_tag = soup.find('h1', class_='tw-text-center')
            if h1_tag:
                data['category_1_title'] = h1_tag.get_text(strip=True)
        else:
            print(f"Skipping Title: Received Status {response.status_code} for {url}")

    except Exception as e:
        print(f"Network/Request Error at {url}: {e}")
        data['status_code'] = "Error"

    return data

def get_product_level_1():
    sitemap_url = "https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_CategoryLv1.xml"
    print("Fetching Category Level 1 Sitemap...")

    try:
        response = requests.get(sitemap_url)
        root = etree.fromstring(response.content)
        namespaces = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        all_urls = root.xpath('//ns:loc/text()', namespaces=namespaces)
    except Exception as e:
        print(f"Failed to read sitemap: {e}")
        return

    # Filter product_type_id 1 and 3
    target_urls = [u for u in all_urls if "product_type_id=1" in u or "product_type_id=3" in u]
    print(f"Total target Level 1 URLs: {len(target_urls)}")

    results = []

    with requests.Session() as session:
        for i, url in enumerate(target_urls):
            print(f"[{i+1}/{len(target_urls)}] Scraping: {url}")

            data = get_category_lv1_title(url, session)
            results.append(data)

            time.sleep(1) 

    df = pd.DataFrame(results)
    return df

In [12]:
cat_lv1_df = get_product_level_1()

Fetching Category Level 1 Sitemap...
Total target Level 1 URLs: 42
[1/42] Scraping: https://www.naiin.com/category?category_1_code=1&product_type_id=1
[2/42] Scraping: https://www.naiin.com/category?category_1_code=2&product_type_id=1
[3/42] Scraping: https://www.naiin.com/category?category_1_code=3&product_type_id=1
[4/42] Scraping: https://www.naiin.com/category?category_1_code=4&product_type_id=1
[5/42] Scraping: https://www.naiin.com/category?category_1_code=5&product_type_id=1
[6/42] Scraping: https://www.naiin.com/category?category_1_code=8&product_type_id=1
[7/42] Scraping: https://www.naiin.com/category?category_1_code=9&product_type_id=1
[8/42] Scraping: https://www.naiin.com/category?category_1_code=10&product_type_id=1
[9/42] Scraping: https://www.naiin.com/category?category_1_code=11&product_type_id=1
[10/42] Scraping: https://www.naiin.com/category?category_1_code=12&product_type_id=1
[11/42] Scraping: https://www.naiin.com/category?category_1_code=13&product_type_id=1
[12

In [13]:
cat_lv1_df.head(5)

,product_type_id,category_1_code,category_1_title,status_code,url
0,1,1,หนังสือพระราชนิพนธ์,200,https://www.naiin.com/category?category_1_code...
1,1,2,นิยาย,200,https://www.naiin.com/category?category_1_code...
2,1,3,หนังสือวาย ยูริ,200,https://www.naiin.com/category?category_1_code...
3,1,4,หนังสือท่องเที่ยว,200,https://www.naiin.com/category?category_1_code...
4,1,5,บ้านและสวน,200,https://www.naiin.com/category?category_1_code...


In [14]:
cat_lv1_df.to_csv('prod_cat_lv1.csv', index=False, encoding='utf-8-sig')

## Category level 2

In [15]:
def get_category_lv2_title(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'th-TH,th;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.naiin.com/',
    }

    parsed_url = urlparse(url)
    params = parse_qs(parsed_url.query)
    p_id = params.get('product_type_id', [''])[0]
    cat1_code = params.get('category_1_code', [''])[0]
    cat2_code = params.get('categoryLv2Code', [''])[0]

    data = {
        'product_type_id': p_id,
        'category_1_code': cat1_code,
        'category_2_code': cat2_code,
        'category_2_title': "-",
        'status_code': None,
        'url': url
    }

    try:
        response = session.get(url, headers=headers, timeout=15)
        data['status_code'] = response.status_code

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            h1_tag = soup.find('h1', class_='tw-text-center')
            if h1_tag:
                data['category_2_title'] = h1_tag.get_text(strip=True)
        else:
            print(f"  [!] Status {response.status_code} for {url}")

    except Exception as e:
        print(f"  [!] Error at {url}: {e}")
        data['status_code'] = "Error"

    return data

def get_product_level_2():
    sitemap_url = "https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_CategoryLv2.xml"
    print("Fetching Category Level 2 Sitemap...")

    try:
        response = requests.get(sitemap_url)
        root = etree.fromstring(response.content)
        namespaces = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        all_urls = root.xpath('//ns:loc/text()', namespaces=namespaces)
    except Exception as e:
        print(f"Failed to read sitemap: {e}")
        return None

    target_urls = [u for u in all_urls if "product_type_id=1" in u or "product_type_id=3" in u]
    print(f"Total target URLs: {len(target_urls)}")

    results = []

    with requests.Session() as session:
        for i, url in enumerate(target_urls):
            print(f"[{i+1}/{len(target_urls)}] Scraping: {url}")

            data = get_category_lv2_title(url, session)
            results.append(data)

            time.sleep(1) 

    df = pd.DataFrame(results)

    return df

In [16]:
cat_lv2_df = get_product_level_2()

Fetching Category Level 2 Sitemap...
Total target URLs: 157
[1/157] Scraping: https://www.naiin.com/category?category_1_code=1&categoryLv2Code=54&product_type_id=1
[2/157] Scraping: https://www.naiin.com/category?category_1_code=1&categoryLv2Code=135&product_type_id=1
[3/157] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=6&product_type_id=1
[4/157] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=7&product_type_id=1
[5/157] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=8&product_type_id=1
[6/157] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=86&product_type_id=1
[7/157] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=134&product_type_id=1
[8/157] Scraping: https://www.naiin.com/category?category_1_code=3&categoryLv2Code=95&product_type_id=1
[9/157] Scraping: https://www.naiin.com/category?category_1_code=3&categoryLv2Code=140&product_type_id=1
[10/

In [17]:
cat_lv2_df.to_csv('prod_cat_lv2.csv', index=False, encoding='utf-8-sig')